# JN08 — Tsunami Inundation Maps

**HySEALab · Postprocessing Notebooks · EDANYA Research Group, Universidad de Málaga**
*Edited by José Manuel González Vida*

---

This notebook post-processes the **inundation output** of any Tsunami-HySEA scenario
(a NetCDF file with `max_height` + `original_bathy`, typically the finest nested level)
and computes the on-land inundation depth.

### Requirements

**Python packages** (any recent version):

```bash
conda install -c conda-forge numpy matplotlib netcdf4 cartopy ipywidgets pillow
# optional: pip install hdf5plugin   (extra HDF5 compression filters)
```

- `cartopy` — map projections and satellite base tiles
- **Internet access is only needed for the satellite map (Step 4)** — every other
  step works fully offline.

**Input data:** a Tsunami-HySEA NetCDF output file containing at least
`max_height` and `original_bathy`. Use the **graphical browser in the first code
cell** to navigate to your simulations folder and pick the `.nc` file. The cell
size (and therefore the flooded-area statistics) is derived automatically from
the grid spacing, so the notebook works for any resolution (10 m, 30 m, …) and
any scenario.

### What you will learn

| Step | Task |
|------|---------|
| 1 | Select the simulation file with a **graphical browser** and load the NetCDF |
| 2 | Understand the **sign convention** for HySEA NC files and compute the **inundation depth** on land |
| 3 | Quick static map (fast, no internet needed) |
| 4 | Full map with **satellite base tiles** (Esri World Imagery) |
| 5 | Statistics: flooded area by depth class, max run-up, depth histogram and run-up diagram |


In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")  # avoids 'NetCDF: HDF error' on network drives
try:
    import hdf5plugin  # registers extra HDF5 filters (zstd/blosc/lz4...); harmless if missing
except Exception:
    pass


# ── Imports ───────────────────────────────────────────────────────────────────
import os
from pathlib import Path

import numpy as np                     # array operations
import matplotlib.pyplot as plt        # plotting
import matplotlib.colors as mcolors    # colour map helpers
import matplotlib.ticker as mticker    # custom tick formatters
from netCDF4 import Dataset            # read Tsunami-HySEA NetCDF output files
import cartopy.crs as ccrs             # coordinate reference systems
import cartopy.feature as cfeature     # built-in geographic features
import cartopy.io.img_tiles as cimgt   # background tile imagery (satellite, OSM)
import ipywidgets as widgets           # interactive file selector
from IPython.display import display

# ── Graphical simulation selector ────────────────────────────────────────────
# Navigate to your simulations folder, select a .nc file and click
# "Use this simulation". This defines NC_FILE, DATA_DIR and SCENARIO, used by
# the rest of the cells. Works with any case / resolution.

NC_FILE = None
DATA_DIR = None
SCENARIO = None

_start = Path.home()
_W = widgets.Layout(width="760px")

w_cwd = widgets.Text(value=str(_start), description="Folder:", layout=_W)
w_go = widgets.Button(description="Go", button_style="info")
w_up = widgets.Button(description="Up", button_style="info")
w_home = widgets.Button(description="Home", button_style="info")
w_refresh = widgets.Button(description="Refresh", button_style="info")
w_list = widgets.Select(options=[], rows=16, layout=_W)
w_use = widgets.Button(description="✓ Use this simulation", button_style="success")
w_sel = widgets.Text(value="", description="Simulation:", layout=_W, disabled=True)
w_status = widgets.HTML(value="")

_state = {"cwd": _start, "busy": False}

def _entries(cwd):
    items = []
    try:
        for p in sorted(cwd.iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
            if p.is_dir():
                items.append((f"📁 {p.name}/", f"D::{p.name}"))
            elif p.suffix.lower() == ".nc":
                items.append((f"🧾 {p.name}", f"F::{p.name}"))
    except PermissionError:
        pass
    return items

def _refresh(*_):
    _state["busy"] = True
    w_cwd.value = str(_state["cwd"])
    opts = _entries(_state["cwd"])
    w_list.options = opts if opts else [("(no folders or .nc files)", "NONE")]
    _state["busy"] = False

def _sel_path():
    val = w_list.value
    if not val or val == "NONE":
        return None, None
    kind, name = val.split("::", 1)
    return kind, _state["cwd"] / name

def _on_pick(change):
    if change.get("name") != "value" or _state["busy"]:
        return
    kind, target = _sel_path()
    if kind == "D" and target is not None:
        _state["cwd"] = target
        _refresh()

def _go(_):
    p = Path(w_cwd.value.strip()).expanduser()
    if p.is_dir():
        _state["cwd"] = p
        _refresh()
    else:
        w_status.value = f"<span style='color:red'>Invalid folder: {p}</span>"

def _up(_):
    _state["cwd"] = _state["cwd"].parent
    _refresh()

def _home(_):
    _state["cwd"] = Path.home()
    _refresh()

def _use(_):
    global NC_FILE, DATA_DIR, SCENARIO
    kind, target = _sel_path()
    if kind != "F" or target is None:
        w_status.value = "<span style='color:red'>Select a .nc file (not a folder).</span>"
        return
    NC_FILE = str(target)
    DATA_DIR = str(target.parent)
    SCENARIO = target.stem
    w_sel.value = NC_FILE
    w_status.value = (f"<span style='color:green'><b>Simulation:</b> {target.name}<br>"
                      f"DATA_DIR = {DATA_DIR}<br>SCENARIO = {SCENARIO}</span>")

w_list.observe(_on_pick, names="value")
w_go.on_click(_go)
w_up.on_click(_up)
w_home.on_click(_home)
w_refresh.on_click(_refresh)
w_use.on_click(_use)

_refresh()
display(widgets.VBox([
    widgets.HTML("<b>Navigate to your simulations folder, select a .nc file and click “Use this simulation”.</b>"),
    widgets.HBox([w_home, w_up, w_refresh]),
    widgets.HBox([w_cwd, w_go]),
    w_list,
    widgets.HBox([w_use]),
    w_sel,
    w_status,
]))


---
## Step 1 — Load the NetCDF Output

The Level-3 NetCDF file contains:

| Variable | Shape | Description |
|---|---|---|
| `lon` | (nx,) | Longitude (°E) |
| `lat` | (ny,) | Latitude (°N) |
| `original_bathy` | (ny, nx) | Pre-seismic bathymetry/topography |
| `deformed_bathy` | (ny, nx) | Post-seismic (coseismic deformation applied) |
| `max_height` | (ny, nx) | Maximum water surface elevation η_max (m, above MSL) |
| `time` | (2,) | Start and end time of simulation (s) |

In [ ]:

# ── Load variables from the NetCDF file ───────────────────────────────────────
if not NC_FILE or not os.path.exists(NC_FILE):
    raise RuntimeError("Select a simulation in the first code cell "
                       "(button \"Use this simulation\").")

with Dataset(NC_FILE) as fl:
    lon            = np.asarray(fl.variables['lon'][:],            dtype=float)  # longitude (°E)
    lat            = np.asarray(fl.variables['lat'][:],            dtype=float)  # latitude  (°N)
    original_bathy = np.asarray(fl.variables['original_bathy'][:], dtype=float)  # pre-seismic bathy/topo (m)
    deformed_bathy = np.asarray(fl.variables['deformed_bathy'][:], dtype=float)  # post-seismic
    max_height     = np.asarray(fl.variables['max_height'][:],     dtype=float)  # eta_max (m, above MSL)
    time_vec       = np.asarray(fl.variables['time'][:],           dtype=float)  # [t_start, t_end] (s)

# 2-D coordinate grids for pcolormesh / contourf
LON, LAT = np.meshgrid(lon, lat)

# ── Cell size derived from the grid (works for ANY resolution / any case) ─────
# Longitude degrees shrink with latitude (× cosφ); latitude degrees are ~constant.
dlon = float(np.abs(np.diff(lon)).mean())
dlat = float(np.abs(np.diff(lat)).mean())
DX_M = dlon * 111320.0 * np.cos(np.deg2rad(float(np.mean(lat))))
DY_M = dlat * 111320.0
CELL_AREA_M2 = DX_M * DY_M

# ── Quick summary ─────────────────────────────────────────────────────────────
print(f'Scenario    : {SCENARIO}')
print(f'Grid size   : {len(lon)} x {len(lat)}  ({len(lon)*len(lat)/1e6:.2f} M cells)')
print(f'Longitude   : [{lon.min():.4f}, {lon.max():.4f}] E')
print(f'Latitude    : [{lat.min():.4f}, {lat.max():.4f}] N')
print(f'Cell size   : dx ~ {DX_M:.1f} m, dy ~ {DY_M:.1f} m  ->  area ~ {CELL_AREA_M2:.0f} m2/cell')
print(f'Sim time    : {time_vec[0]:.0f} - {time_vec[-1]:.0f} s  ({time_vec[-1]/3600:.1f} h)')
print()
print(f'original_bathy : min = {original_bathy.min():8.3f}  max = {original_bathy.max():.3f} m')
print(f'max_height     : min = {max_height.min():8.3f}  max = {max_height.max():.3f} m')


---
## Step 2 — Sign Convention and Inundation Depth

### ⚠️ HySEA NC output sign convention

| File / variable | Positive | Negative |
|---|---|---|
| **GRD input** (`.grd`) | land elevation | ocean depth |
| **NC output** (`original_bathy`) | ocean depth | land elevation |

The NC sign is **inverted** with respect to the GMT/GEBCO convention used in input grids.

### Inundation depth formula

On a land cell (where `original_bathy < 0`, elevation above MSL = `|original_bathy|`):

$$d_{inun} = \eta_{max} + z_{bathy}$$

where $z_{bathy} = $ `original_bathy` (negative on land).  
This gives the **depth of water above the ground surface** at peak flood.

| Example | `original_bathy` | `max_height` | `inundation_depth` |
|---|---|---|---|
| Flat 2 m land, wave reaches 5 m | −2 m | 5 m | **3 m** |
| Elevated 4 m land, wave reaches 3 m | −4 m | 3 m | −1 m → **no flood** |
| Ocean cell (not land) | +50 m | 2.5 m | not applicable |

We set `inundation_depth = NaN` wherever:
- the cell is ocean (`original_bathy ≥ 0`), OR
- the wave did not reach the cell (`inundation_depth ≤ 0`)

In [ ]:

# ── Compute inundation depth ──────────────────────────────────────────────────
# HySEA NC sign convention (opposite to the GMT/.grd input convention):
#   original_bathy >= 0  ->  ocean (depth below sea level, expressed as positive)
#   original_bathy <  0  ->  land  (elevation above MSL, expressed as negative)

land_mask  = original_bathy < 0           # True on dry land
flood_raw  = max_height + original_bathy   # raw water depth above ground
flood_mask = land_mask & (flood_raw > 0)   # land AND actually flooded

inundation_depth = np.full_like(original_bathy, fill_value=np.nan, dtype=float)
inundation_depth[flood_mask] = flood_raw[flood_mask]

# ── Summary statistics ────────────────────────────────────────────────────────
n_flooded  = flood_mask.sum()
cell_area  = CELL_AREA_M2                    # ACTUAL cell area (derived from the grid spacing above)
flood_area = n_flooded * cell_area / 1e6     # m² -> km²

print(f'Land cells     : {land_mask.sum():10,}')
print(f'Flooded cells  : {n_flooded:10,}  ({100*n_flooded/land_mask.sum():.1f}% of land)')
print(f'Flooded area   : {flood_area:.4f} km²  ({flood_area*1e6:.0f} m²)')
print(f'Max inundation : {np.nanmax(inundation_depth):.2f} m')
print(f'Mean inundation: {np.nanmean(inundation_depth):.2f} m  (flooded cells only)')


---
### Embedded HySEA logo (optional)

The Tsunami-HySEA logo is **embedded in the notebook** (base64), so it does not
depend on any external file. Its **black background is made transparent** and it
is placed, **small and unobtrusive**, in the map corner with **the largest water
surface** (detected with `original_bathy ≥ 0`), so it never covers the flooded area.


In [ ]:
# ── Embedded HySEA logo (base64) + helpers to place it on the map ────────────
import base64, io
from PIL import Image

LOGO_B64 = (
    "..."
)


def load_hysea_logo(threshold=40, transparent_bg=True):
    """Decode the embedded logo.

    transparent_bg=True  -> transparent black background (ideal on DARK
                            backgrounds, e.g. the Step-4 satellite imagery).
    transparent_bg=False -> OPAQUE logo with its black background (ideal on
                            LIGHT backgrounds, where white text would vanish).
    """
    img = Image.open(io.BytesIO(base64.b64decode(LOGO_B64))).convert("RGBA")
    arr = np.array(img)
    if transparent_bg:
        black = arr[..., :3].astype(int).sum(axis=2) < threshold   # nearly-black pixels
        arr[black, 3] = 0                                          # -> alpha 0 (transparent)
    return arr


def best_water_corner(lon, lat, original_bathy, frac=0.30):
    """Return the corner ('top-right'...) with the LARGEST ocean fraction.

    NC convention: original_bathy >= 0 is ocean. A block of size 'frac' is
    examined at each geographic corner and the one with the most water is
    chosen (so the logo never covers the inundated area).
    """
    LONc, LATc = np.meshgrid(lon, lat)
    ocean = original_bathy >= 0
    dlon, dlat = lon.max() - lon.min(), lat.max() - lat.min()
    west  = LONc <= lon.min() + frac * dlon
    east  = LONc >= lon.max() - frac * dlon
    south = LATc <= lat.min() + frac * dlat
    north = LATc >= lat.max() - frac * dlat
    corners = {"top-left": north & west, "top-right": north & east,
               "bottom-left": south & west, "bottom-right": south & east}
    scores = {k: (float(ocean[m].mean()) if m.any() else 0.0) for k, m in corners.items()}
    return max(scores, key=scores.get), scores


def add_logo_corner(fig, ax, logo_rgba, corner="top-right",
                    width_frac=0.25, alpha=1.0, margin=0.0,
                    margin_x=None, margin_y=None):
    """Place the (small) logo in a corner of the axes, preserving its aspect.

    margin_x / margin_y adjust the horizontal and vertical separation from the
    border independently (as a fraction of the axes). They default to `margin`.
    """
    mx = margin if margin_x is None else margin_x
    my = margin if margin_y is None else margin_y
    aspect = logo_rgba.shape[1] / logo_rgba.shape[0]
    # ACTUAL axes aspect ratio (without redrawing, to avoid triggering tile downloads)
    try:
        p = ax.get_position()
        fw, fh = fig.get_size_inches()
        axes_ratio = (p.width * fw) / (p.height * fh)
    except Exception:
        fw, fh = fig.get_size_inches()
        axes_ratio = fw / fh
    h_frac = width_frac * axes_ratio / aspect
    x_left, x_right = mx, 1 - mx - width_frac
    y_bottom, y_top = my, 1 - my - h_frac
    pos = {"top-left": (x_left, y_top), "top-right": (x_right, y_top),
           "bottom-left": (x_left, y_bottom), "bottom-right": (x_right, y_bottom)}[corner]
    # Multiply alpha onto the existing channel (do NOT overwrite it) so the
    # transparent black background is preserved even with alpha=1.
    logo = logo_rgba.copy()
    if alpha is not None and alpha < 1.0:
        logo[..., 3] = (logo[..., 3].astype(float) * float(alpha)).astype(logo.dtype)
    lax = ax.inset_axes([pos[0], pos[1], width_frac, h_frac])
    lax.imshow(logo)
    lax.axis("off")
    lax.set_zorder(10)
    return lax

print("HySEA logo loaded and helpers ready.")


---
## Step 3 — Quick Static Map

A fast overview plot without satellite tiles — useful when internet access is limited or slow.

In [ ]:

VMAX = 5.0   # cap the colour scale at 5 m so deep floods share the darkest colour

# ── Animated progress indicator (same as in Step 4) ─────────────────────────
import threading, itertools, time
_spin3 = widgets.HTML(value="")
display(_spin3)
_spin3_stop = {"flag": False}

def _run_spinner3(msg):
    for ch in itertools.cycle("\u280b\u2819\u2839\u2838\u283c\u2834\u2826\u2827\u2807\u280f"):
        if _spin3_stop["flag"]:
            break
        _spin3.value = f"<span style='font-size:15px'>{ch} {msg}</span>"
        time.sleep(0.12)

_spin3_thread = threading.Thread(
    target=_run_spinner3,
    args=("Step 3: drawing the static map\u2026",),
    daemon=True,
)
_spin3_thread.start()

fig, ax = plt.subplots(1, 1, figsize=(7, 12),
                        subplot_kw={'projection': ccrs.Mercator()})

# ── Neutral topobathymetry background ────────────────────────────────────────
# To AVOID confusion with the warm inundation colour scale (YlOrRd), the base map
# uses NEUTRAL tones: cool pale blue for the ocean and GREYSCALE for the land.
# This keeps all warm colours (yellow->red) exclusively for the inundation layer.
ocean_depth = np.where(original_bathy >= 0,  original_bathy, np.nan)  # sea cells (depth)
land_elev   = np.where(original_bathy <  0, -original_bathy, np.nan)  # land cells (elevation > 0)

# Ocean: pale, low-saturation blue (cool tone, does not clash with warm scale)
ax.pcolormesh(LON, LAT, ocean_depth, transform=ccrs.PlateCarree(),
              cmap='Blues', vmin=0, vmax=200, alpha=0.45)
# Land: greyscale relief (neutral) instead of 'terrain' green/brown
_land_vmax = float(np.nanmax(land_elev)) if np.isfinite(land_elev).any() else 1.0
ax.pcolormesh(LON, LAT, land_elev, transform=ccrs.PlateCarree(),
              cmap='Greys', vmin=0, vmax=max(_land_vmax, 1.0), alpha=0.55)

# ── Inundation layer (the only warm colours on the map) ───────────────────────
inund_plot = np.clip(inundation_depth, 0, VMAX)
levels = np.linspace(0.1, VMAX, 20)
cs = ax.contourf(LON, LAT, inund_plot, levels=levels,
                 transform=ccrs.PlateCarree(),
                 cmap='YlOrRd', extend='max', alpha=0.90)

cbar = fig.colorbar(cs, ax=ax, orientation='vertical', fraction=0.04, pad=0.02)
cbar.set_label('Inundation depth (m)', fontsize=12)

# ── Coastline = 0 m isocontour of the model bathymetry (grid-aligned) ────────
ax.contour(LON, LAT, original_bathy, levels=[0],
           transform=ccrs.PlateCarree(),
           colors='black', linewidths=0.8, alpha=0.9)

# ── Coordinate grid lines ─────────────────────────────────────────────────────
gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='gray', alpha=0.6,
                  x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False

ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
ax.set_title(f'Inundation depth — {SCENARIO}', fontsize=13)


# ── HySEA logo in the corner with the most water ─────────────────────────────
try:
    add_logo_corner(fig, ax, load_hysea_logo(transparent_bg=False),
                    corner=best_water_corner(lon, lat, original_bathy)[0],
                    width_frac=0.25, alpha=1.0, margin_x=0.0, margin_y=0.04)
except Exception as _e:
    print(f'Note: logo not added ({type(_e).__name__}: {_e}).')

plt.tight_layout()
try:
    fig.canvas.draw()   # force the render under the spinner (Step 3 downloads no tiles)
finally:
    _spin3_stop["flag"] = True
    _spin3_thread.join(timeout=1.0)
    _spin3.value = "<span style='color:green'>\u2713 Step 3 completed.</span>"
plt.show()


---
## Step 4 — Map with Satellite Base Tiles

We add a Google Maps satellite image as background using `cartopy.io.img_tiles.GoogleTiles`.  
The zoom level controls the tile resolution:

| Zoom | Approx. tile resolution | Recommended for |
|------|------------------------|------------------|
| 13   | ~20 m/px               | city overview |
| 15   | ~5 m/px                | street level |
| 17   | ~1 m/px                | building level (slow) |

For a 10 m grid, **zoom = 14–15** gives a good balance.  
The inundation layer is drawn on top with `alpha=0.75` so the satellite is visible.

> **Note:** Tile downloads require internet access. If unavailable, use the static map from Step 3.

In [ ]:

# ── Step 4 — satellite map (robust against missing tiles / 404 floods) ────────
USE_SATELLITE = True   # False -> downloads NO tiles: neutral background (offline mode)
ZOOM = 14              # if tiles are missing at this zoom, lower it to 12-13
VMAX = 5.0             # colour scale cap (m), same as the static map

import io, contextlib, logging, warnings
from IPython.display import Image as IPImage, display

# Lower the log level of the network modules (just in case)
for _n in ('cartopy', 'cartopy.io', 'cartopy.io.img_tiles',
           'urllib3', 'urllib3.connectionpool', 'PIL'):
    logging.getLogger(_n).setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='.*[Tt]ile.*')

# RELIABLE satellite source: Esri World Imagery (no API key needed).
# The default Google 'satellite' source often returns 404 (hence the flood of
# messages). Esri World Imagery has good coverage and requires no key.
class _EsriImagery(cimgt.GoogleTiles):
    def _image_url(self, tile):
        x, y, z = tile
        return ('https://server.arcgisonline.com/ArcGIS/rest/services/'
                'World_Imagery/MapServer/tile/%d/%d/%d' % (z, y, x))

if USE_SATELLITE:
    try:
        imagery = _EsriImagery()
    except Exception:
        imagery = cimgt.OSM()
    proj = imagery.crs
else:
    imagery = None
    proj = ccrs.Mercator()

fig, ax = plt.subplots(1, 1, figsize=(8, 14), subplot_kw={'projection': proj})
ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())

# Neutral background ALWAYS underneath: if tiles fail to load, the map stays legible
ax.add_feature(cfeature.OCEAN, facecolor='#dfe7ee', zorder=0)
ax.add_feature(cfeature.LAND,  facecolor='#eceae4', zorder=0)

# Add the satellite layer. cartopy prints "HTTP Error 404" to stdout when a
# tile does not exist; we capture it so it does NOT flood the output.
if imagery is not None:
    _buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(_buf), contextlib.redirect_stderr(_buf):
            ax.add_image(imagery, ZOOM)
    except Exception as e:
        print(f'Note: could not prepare the satellite layer ({type(e).__name__}); '
              f'using the neutral background.')

# ── Inundation layer on top ───────────────────────────────────────────────────
inund_plot = np.clip(inundation_depth, 0, VMAX)
levels = [0.1, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

cs = ax.contourf(LON, LAT, inund_plot, levels=levels,
                 transform=ccrs.PlateCarree(),
                 cmap='YlOrRd', extend='max', alpha=0.75)

ax.contour(LON, LAT, inund_plot, levels=[0.1, 1.0, 3.0],
           transform=ccrs.PlateCarree(),
           colors=['white', 'orange', 'red'], linewidths=[0.5, 0.8, 1.0], alpha=0.9)

cbar = fig.colorbar(cs, ax=ax, orientation='vertical', fraction=0.04, pad=0.02,
                    ticks=levels)
cbar.set_label('Inundation depth (m)', fontsize=12)
cbar.ax.set_yticklabels([f'{v:.1f}' for v in levels])

gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='white', alpha=0.6,
                  x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xformatter = mticker.FuncFormatter(lambda v, _: f'{v:.3f}°E')
gl.yformatter = mticker.FuncFormatter(lambda v, _: f'{v:.3f}°N')
ax.set_title(f'Inundation depth — {SCENARIO}\n'
             f'colour scale capped at {VMAX} m', fontsize=12)


# ── HySEA logo in the corner with the most water (small, unobtrusive) ───────
try:
    _logo = load_hysea_logo()
    _corner, _scores = best_water_corner(lon, lat, original_bathy)
    add_logo_corner(fig, ax, _logo, corner=_corner, width_frac=0.25, alpha=1.0,
                    margin_x=0.0, margin_y=0.04)
    print(f'HySEA logo corner: {_corner}  (water ~ {_scores[_corner]*100:.0f}%)')
except Exception as e:
    print(f'Note: could not add the logo ({type(e).__name__}: {e}).')

plt.tight_layout()

# Tiles are downloaded at DRAW time (savefig). We do it once and silently;
# then we close the figure to avoid a second inline draw (which would repeat
# the flood of 404s) and display the already-generated PNG.
# ── Animated progress indicator (Step 4 is usually the slowest) ─────────────
import threading, itertools, time
_spin = widgets.HTML(value="")
display(_spin)
_spin_stop = {"flag": False}

def _run_spinner(msg):
    for ch in itertools.cycle("\u280b\u2819\u2839\u2838\u283c\u2834\u2826\u2827\u2807\u280f"):
        if _spin_stop["flag"]:
            break
        _spin.value = f"<span style='font-size:15px'>{ch} {msg}</span>"
        time.sleep(0.12)

_spin_thread = threading.Thread(
    target=_run_spinner,
    args=("Step 4: downloading satellite tiles and drawing the map\u2026",),
    daemon=True,
)
_spin_thread.start()

# Tiles are downloaded at DRAW time (savefig). Once and silently; then we
# close the figure to avoid a second inline draw and display the PNG.
_out_map = os.path.join(DATA_DIR, f'{SCENARIO}_inundation_map.png')
_buf2 = io.StringIO()
try:
    with contextlib.redirect_stdout(_buf2), contextlib.redirect_stderr(_buf2):
        fig.savefig(_out_map, dpi=150, bbox_inches='tight')
finally:
    _spin_stop["flag"] = True
    _spin_thread.join(timeout=1.0)
    _spin.value = "<span style='color:green'>\u2713 Step 4 completado.</span>"

plt.close(fig)
print(f'Map saved: {_out_map}')
display(IPImage(filename=_out_map))


---
## Step 5 — Statistics: Flooded Area, Depth Distribution

Quantitative summary of the inundation result:  
- Flooded area by depth class  
- Maximum run-up elevation (max `max_height` on land)  
- Depth histogram

In [ ]:

# ── Inundation statistics by depth class ─────────────────────────────────────
# CELL_AREA_M2 is computed in cell 3 from the actual grid spacing (any resolution).

depth_classes = [(0.0, 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 3.0), (3.0, 5.0), (5.0, np.inf)]
labels        = ['0.0-0.5 m', '0.5-1.0 m', '1.0-2.0 m', '2.0-3.0 m', '3.0-5.0 m', '> 5.0 m']

valid = inundation_depth[flood_mask]   # 1-D depths of flooded cells only

header = f"{'Depth class':<15}  {'Cells':>8}  {'Area (m2)':>12}  {'Area (km2)':>12}  {'%':>6}"
print(header)
print('─' * 65)
for (lo, hi), lbl in zip(depth_classes, labels):
    mask = (valid >= lo) & (valid < hi)
    n    = mask.sum()
    area = n * CELL_AREA_M2
    pct  = 100 * n / len(valid) if len(valid) > 0 else 0
    print(f'{lbl:<15}  {n:>8,}  {area:>12,.0f}  {area/1e6:>12.4f}  {pct:>5.1f}%')
print('─' * 65)
total_n    = len(valid)
total_area = total_n * CELL_AREA_M2
print(f"{'TOTAL':<15}  {total_n:>8,}  {total_area:>12,.0f}  {total_area/1e6:>12.4f}  100.0%")
print()

# ── Maximum run-up (max eta_max on land) ─────────────────────────────────────
runup_max = float(max_height[land_mask].max())
runup_loc = np.unravel_index(np.argmax(np.where(land_mask, max_height, -np.inf)), max_height.shape)
print(f'Max run-up (eta_max on land) : {runup_max:.2f} m')
print(f'  at lon = {lon[runup_loc[1]]:.4f} E, lat = {lat[runup_loc[0]]:.4f} N')


In [ ]:

# ── Two-panel diagnostic figure: depth histogram + run-up diagram ────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: depth histogram
ax0 = axes[0]
bins = np.linspace(0, min(VMAX + 1, valid.max() + 0.5), 40)
ax0.hist(valid, bins=bins, color='tomato', edgecolor='white', linewidth=0.4)
ax0.set_xlabel('Inundation depth (m)', fontsize=11)
ax0.set_ylabel('Number of cells', fontsize=11)
ax0.set_title(f'Distribution of inundation depths\n{SCENARIO}', fontsize=11)
ax0.axvline(valid.mean(), color='navy', linestyle='--', linewidth=1.5,
            label=f'Mean = {valid.mean():.2f} m')
ax0.axvline(np.median(valid), color='darkorange', linestyle=':', linewidth=1.5,
            label=f'Median = {np.median(valid):.2f} m')
ax0.legend(fontsize=10)
ax0.grid(axis='y', alpha=0.4)

# Right: run-up diagram (land elevation vs eta_max for each flooded cell)
ax1 = axes[1]
elev_land = -original_bathy[flood_mask]   # land elevation (positive)
mh_land   = max_height[flood_mask]         # eta_max at flooded land cells
ax1.scatter(elev_land, mh_land, s=0.2, c='steelblue', alpha=0.3, rasterized=True)
xlim = [0, min(30, elev_land.max() * 1.1)]
ax1.plot(xlim, xlim, 'k--', linewidth=1, label='eta_max = elevation (limit of flooding)')
ax1.set_xlabel('Land elevation above MSL (m)', fontsize=11)
ax1.set_ylabel('eta_max — max water surface (m)', fontsize=11)
ax1.set_title('Run-up diagram\n(each point = one flooded cell)', fontsize=11)
ax1.set_xlim(xlim)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.4)


plt.tight_layout()
_out_stats = os.path.join(DATA_DIR, f'{SCENARIO}_inundation_stats.png')
plt.savefig(_out_stats, dpi=120, bbox_inches='tight')
print(f'Stats figure saved: {_out_stats}')
plt.show()


---
### How to read the run-up diagram

Each point in the scatter plot represents a single **flooded grid cell**.

| Axis | Meaning |
|---|---|
| **X — Land elevation above MSL (m)** | Terrain height *before* the tsunami (pre-seismic topography) |
| **Y — η_max (m)** | Maximum water surface elevation reached at that cell during the simulation |

**The dashed diagonal line (y = x)** is the theoretical flooding limit: a point on the line means the water surface just reached the ground level with zero depth.  
Every point must lie **above** the diagonal — if η_max > elevation, there is water on the ground.

**Reading the cloud of points:**

| Region | Interpretation |
|---|---|
| Points near the diagonal (lower-left corner) | Inland flooding limit — water barely reached these cells, inundation depth ≈ 0 |
| Dense cloud at x = 0–2 m, y = 5–8 m | Large number of low, flat coastal cells where the wave reached ~6 m above MSL — the most severely flooded area |
| Points at x = 4–8 m close to the diagonal | Higher terrain flooded with little water depth (marginal inundation) |
| Visible horizontal bands | Rows of cells at a constant terrain level (streets, platforms) where η_max is nearly uniform |

The inundation depth at any cell is simply the **vertical distance from the diagonal**:

$$d_{inun} = \eta_{max} - \text{elevation} = \eta_{max} + z_{bathy}$$

In the example shown (Catania, 10 m grid), the concentration of points at x < 2 m confirms that most of the inundated area is nearly flat, low-lying coastal terrain.

---
## Summary

| Step | Done | Output |
|------|------|--------|
| Select + load NC | ✓ | graphical selector → `lon`, `lat`, `original_bathy`, `max_height` |
| Sign convention | ✓ | NC: `+` = ocean, `−` = land  (inverted vs GRD input) |
| Inundation depth | ✓ | `inundation_depth = max_height + original_bathy`  (land only, > 0) |
| Static map | ✓ | neutral basemap (grey land + pale-blue sea) + warm inundation |
| Satellite map | ✓ | `<SCENARIO>_inundation_map.png` |
| Statistics | ✓ | `<SCENARIO>_inundation_stats.png`  +  depth table |

### Notes

- The **cell size** (and so the flooded-area statistics) is derived automatically from
  the grid spacing, so the notebook works for any resolution (10 m, 30 m, …).
- The Step-3 basemap uses **neutral tones** (greyscale land, pale-blue ocean) so the
  warm `YlOrRd` scale is used **only** for the inundation depth and is not confused
  with the topobathymetry.
- Output figures are saved next to the NC file, named after the selected scenario.

### To apply to another case

Just re-run **cell 1** and pick a different `.nc` in the graphical selector — all
the cells below run unchanged.